Get data from worksheet

In [1]:
from tableauscraper import TableauScraper as TS
import pandas as pd
import os

url = "https://dvis3.ddc.moph.go.th/t/DDC_CENTER_DOE/views/priority_v2/Dashboard2?%3Aembed=y&%3AisGuestRedirectFromVizportal=y" 

ts = TS()
ts.loads(url)

workbook = ts.getWorkbook() 

data_list = []
for t in workbook.worksheets:
    print(f"工作表名称: {t.name}")

工作表名称: ตารางรายสัปดาห์
工作表名称: หมายเหตุหน้า 2


In [2]:
output_dir = os.path.join('..', 'Data', 'Tableau')
os.makedirs(output_dir, exist_ok=True)
export_summary = []

# process only the first worksheet
filename = 'all_weekly_priority_cases.csv'

# columns from the prompt (keep these if present)
keep_cols = [
    "ชื่อโรค-value",
    # "ชื่อโรค-alias","MIN(Number of Records)-alias",
    "สัปดาห์-value",
    # "สัปดาห์-alias",
    "ATTR(week_number)-alias",
    "ATTR(disease (copy))-alias",
    # "ATTR(disease_th)-alias",
    "AGG(จำนวนป่วย (แทน null))-alias",
    # "AGG(Formatted Percent Change)-alias","AGG(Arrow Type)-alias","AGG(Consecutive Weeks)-alias"
]

# optional rename map to English-friendly names
rename_map = {
    "ชื่อโรค-value": "disease_value",
    "ชื่อโรค-alias": "disease_alias",
    "MIN(Number of Records)-alias": "min_records_alias",
    "สัปดาห์-value": "week_value",
    "สัปดาห์-alias": "week_alias",
    "ATTR(week_number)-alias": "week_number_alias",
    "ATTR(disease (copy))-alias": "disease_copy_alias",
    "ATTR(disease_th)-alias": "disease_th_alias",
    "AGG(จำนวนป่วย (แทน null))-alias": "cases_alias",
    "AGG(Formatted Percent Change)-alias": "pct_change_alias",
    "AGG(Arrow Type)-alias": "arrow_type_alias",
    "AGG(Consecutive Weeks)-alias": "consecutive_weeks_alias"
}

if not workbook.worksheets:
    print("No worksheets found in workbook.")
    export_summary.append({'name': None, 'rows': 0, 'path': None, 'error': 'No worksheets'})
else:
    t = workbook.worksheets[0]
    print(f'Processing first worksheet: {t.name}')
    try:
        df = t.data
        if df is None:
            print(f'  No data for: {t.name}')
            export_summary.append({'name': t.name, 'filename': filename, 'rows': 0, 'path': None})
        else:
            pdf = pd.DataFrame(df)
            # select only columns that exist in the scraped data
            present = [c for c in keep_cols if c in pdf.columns]
            if not present:
                # fallback: try some common alternatives if column names differ
                print("  None of the expected columns found; saving full sheet as fallback.")
                filtered = pdf.copy()
            else:
                filtered = pdf[present].rename(columns={c: rename_map.get(c, c) for c in present})

            outpath = os.path.join(output_dir, filename)
            # filtered.to_csv(outpath, index=False)
            # print(f'  Saved: {outpath} (rows={len(filtered)})')
            export_summary.append({'name': t.name, 'filename': filename, 'rows': len(filtered), 'path': outpath})
    except Exception as e:
        print(f'  Error processing {t.name}: {e}')
        export_summary.append({'name': t.name, 'filename': filename, 'rows': None, 'path': None, 'error': str(e)})

Processing first worksheet: ตารางรายสัปดาห์


In [9]:
workbook.getParameters()

[{'column': 'ปี',
  'values': ['2568', '2567', '2566', '2565', '2564', '2563'],
  'parameterName': '[Parameters].[Parameter 1]'}]

In [5]:
filters = t.getFilters()
if filters is not None:
    for item in filters:
        print(f'Filter: {item}')

Filter: {'column': 'SKR', 'ordinal': 0, 'values': ['เขตสุขภาพ 1', 'เขตสุขภาพ 2', 'เขตสุขภาพ 3', 'เขตสุขภาพ 4', 'เขตสุขภาพ 5', 'เขตสุขภาพ 6', 'เขตสุขภาพ 7', 'เขตสุขภาพ 8', 'เขตสุขภาพ 9', 'เขตสุขภาพ 10', 'เขตสุขภาพ 11', 'เขตสุขภาพ 12', 'เขตสุขภาพ 13', 'ไม่ระบุ'], 'globalFieldName': '[sqlproxy.0ju0gj407vd1ga16bb3k71585m80].[none:SKR:nk]', 'selection': ['เขตสุขภาพ 1', 'เขตสุขภาพ 2', 'เขตสุขภาพ 3', 'เขตสุขภาพ 4', 'เขตสุขภาพ 5', 'เขตสุขภาพ 6', 'เขตสุขภาพ 7', 'เขตสุขภาพ 8', 'เขตสุขภาพ 9', 'เขตสุขภาพ 10', 'เขตสุขภาพ 11', 'เขตสุขภาพ 12', 'เขตสุขภาพ 13', 'ไม่ระบุ', 'all'], 'selectionAlt': [{'fn': '[sqlproxy.0ju0gj407vd1ga16bb3k71585m80].[none:SKR:nk]', 'columnFullNames': ['[SKR]'], 'domainTables': [{'isSelected': True, 'label': 'เขตสุขภาพ 1'}]}]}
Filter: {'column': 'age_group', 'ordinal': 0, 'values': ['0-4', '5-9', '10-14', '15-19', '20-29', '30-39', '40-49', '50-59', '60+'], 'globalFieldName': '[sqlproxy.0ju0gj407vd1ga16bb3k71585m80].[none:age_group:nk]', 'selection': ['0-4', '5-9', '10-14', '